## Feature generailization

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

def preprocess_with_interactions_selection(
    df,
    target_col="Churn",
    id_cols=None,
    test_size=0.2,
    random_state=42,
    k_features=50  # number of features to keep
):
    df = df.copy()

    # -------------------------
    # 1. Drop ID columns
    # -------------------------
    if id_cols:
        df = df.drop(columns=id_cols, errors="ignore")

    # -------------------------
    # 2. Split target
    # -------------------------
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=test_size, random_state=random_state
    )

    # -------------------------
    # 3. Identify column types
    # -------------------------
    numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
    categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns

    # -------------------------
    # 4. Pipelines
    # -------------------------

    # Numeric: impute → interactions → scale
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(
            degree=2,
            interaction_only=True,
            include_bias=False
        )),
        ("scaler", StandardScaler())
    ])

    # Categorical: impute → one-hot
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    # Combine
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    preprocessor.set_output(transform="pandas")

    # -------------------------
    # 5. Feature Selection (keeps names!)
    # -------------------------
    selector = SelectKBest(score_func=f_classif, k=k_features)

    full_pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("feature_selection", selector)
    ])

    # -------------------------
    # 6. Transform
    # -------------------------
    X_train_processed = full_pipeline.fit_transform(X_train, y_train)
    X_test_processed = full_pipeline.transform(X_test)

    # -------------------------
    # 7. Get selected feature names
    # -------------------------
    feature_names = full_pipeline.named_steps["preprocessing"].get_feature_names_out()
    selected_mask = full_pipeline.named_steps["feature_selection"].get_support()

    selected_features = feature_names[selected_mask]

    # Convert to DataFrame with names
    X_train_processed = pd.DataFrame(X_train_processed, columns=selected_features)
    X_test_processed = pd.DataFrame(X_test_processed, columns=selected_features)

    return (
        X_train_processed,
        X_test_processed,
        y_train.reset_index(drop=True),
        y_test.reset_index(drop=True),
        full_pipeline,
        selected_features
    )

In [9]:
df = pd.read_csv("E Commerce Dataset.csv")

X_train, X_test, y_train, y_test, pipeline, selected_features = \
    preprocess_with_interactions_selection(
        df,
        target_col="Churn",         # change if different
        id_cols=["CustomerID"],     # optional
        test_size=0.2,
        random_state=42,
        k_features=50               # number of features to keep
    )

In [10]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4504 entries, 0 to 4503
Data columns (total 50 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   num__Tenure                                         4504 non-null   float64
 1   num__NumberOfDeviceRegistered                       4504 non-null   float64
 2   num__SatisfactionScore                              4504 non-null   float64
 3   num__Complain                                       4504 non-null   float64
 4   num__DaySinceLastOrder                              4504 non-null   float64
 5   num__CashbackAmount                                 4504 non-null   float64
 6   num__Tenure CityTier                                4504 non-null   float64
 7   num__Tenure WarehouseToHome                         4504 non-null   float64
 8   num__Tenure HourSpendOnApp                          4504 non-null   float64
 9

## Class imbalance

In [11]:
y_train.value_counts()

Churn
0    3746
1     758
Name: count, dtype: int64

In [12]:
# SMOTE for upsampling.

# Upsample by creating new, synthetic data points rather than duplicating existing ones.
# Its main advantages over naive methods like random oversampling include reduced overfitting, improved model generalization, and better preservation of minority class information
# !pip install imbalanced-learn

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

y_train_smote.value_counts()


Churn
0    3746
1    3746
Name: count, dtype: int64

Data set for modelling is X_train_smote, X_test, y_train_smote and y_test


## Logistic Regression

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

logreg = LogisticRegression(solver='liblinear', max_iter=1000)

logreg.fit(X_train_smote, y_train_smote)

y_pred = logreg.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", class_report)

Accuracy: 0.7992895204262878
Confusion Matrix:
 [[742 194]
 [ 32 158]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.79      0.87       936
           1       0.45      0.83      0.58       190

    accuracy                           0.80      1126
   macro avg       0.70      0.81      0.73      1126
weighted avg       0.87      0.80      0.82      1126



## Random Forest

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import recall_score, accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import numpy as np

rf = RandomForestClassifier(random_state=42)

param_dist = {
    'n_estimators': [100, 200, 500, 800, 1000],
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],  # good for 76 features
    'bootstrap': [True, False]
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#Randomized because we have a relatively higher number of features
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,                  # number of random combinations to try
    scoring='accuracy',          # change to 'f1' or 'roc_auc' if needed
    cv=skf,
    verbose=1,
    random_state=42,
    n_jobs=-1                    # use all processors
)

random_search.fit(X_train_smote, y_train_smote)

print("Best Parameters:", random_search.best_params_)
print("Best CV Accuracy:", random_search.best_score_)

best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:,1]



print("\n=== Test Set Evaluation ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 50 candidates, totalling 250 fits


Best Parameters: {'n_estimators': 800, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 50, 'bootstrap': False}
Best CV Accuracy: 0.9779767731224466

=== Test Set Evaluation ===
Accuracy: 0.977797513321492
F1 Score: 0.9333333333333333
ROC AUC: 0.9963618983355825
Confusion Matrix:
 [[926  10]
 [ 15 175]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.99       936
           1       0.95      0.92      0.93       190

    accuracy                           0.98      1126
   macro avg       0.97      0.96      0.96      1126
weighted avg       0.98      0.98      0.98      1126



In [15]:
w_rf = recall_score(y_test, y_pred)

## XGboost

In [16]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import numpy as np

xgb_clf = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

param_dist = {
    'n_estimators': [100, 200, 500, 800],
    'max_depth': [3, 5, 7, 10, 15],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2, 0.3, 0.5],
    'min_child_weight': [1, 3, 5, 7]
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    n_iter=50,                  # number of random combinations to try
    scoring='accuracy',          # can change to 'f1' or 'roc_auc'
    cv=skf,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_smote, y_train_smote)

print("Best Parameters:", random_search.best_params_)
print("Best CV Accuracy:", random_search.best_score_)

best_xgb = random_search.best_estimator_
y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:,1]

print("\n=== Test Set Evaluation ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'subsample': 1.0, 'n_estimators': 800, 'min_child_weight': 3, 'max_depth': 15, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8}
Best CV Accuracy: 0.9818470880898793

=== Test Set Evaluation ===
Accuracy: 0.9866785079928952
F1 Score: 0.9597855227882037
ROC AUC: 0.9956590193432299
Confusion Matrix:
 [[932   4]
 [ 11 179]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99       936
           1       0.98      0.94      0.96       190

    accuracy                           0.99      1126
   macro avg       0.98      0.97      0.98      1126
weighted avg       0.99      0.99      0.99      1126



In [17]:
w_xgb = recall_score(y_test, y_pred)

## LightGBM

In [18]:
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

lgbm = lgb.LGBMClassifier(
    objective='binary',
    random_state=42,
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=5,
    min_child_samples=40,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2
)

param_dist = {
    'n_estimators': [100, 200, 500, 800],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [15, 31, 63, 127],
    'max_depth': [-1, 5, 10, 20],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1',
    cv=skf,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_smote, y_train_smote)

print("Best Parameters:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

best_lgbm = random_search.best_estimator_
y_pred = best_lgbm.predict(X_test)
y_proba = best_lgbm.predict_proba(X_test)[:, 1]

print("\n=== Test Set Evaluation ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 2997, number of negative: 2997[LightGBM] [Info] Number of positive: 2996, number of negative: 2997

[LightGBM] [Info] Number of positive: 2997, number of negative: 2997
[LightGBM] [Info] Number of positive: 2997, number of negative: 2996
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019984 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12229
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing

In [19]:
w_lgbm = recall_score(y_test, y_pred)

## Model ensemble

In [21]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import numpy as np

# -------------------------
# Get probabilities
# -------------------------
rf_proba = best_rf.predict_proba(X_test)[:, 1]
xgb_proba = best_xgb.predict_proba(X_test)[:, 1]
lgbm_proba = best_lgbm.predict_proba(X_test)[:, 1]

recalls = [w_rf, w_xgb, w_lgbm]
total = sum(recalls)

w_rf, w_xgb, w_lgbm = [r / total for r in recalls]

print("Weights:", w_rf, w_xgb, w_lgbm)

# -------------------------
# Weighted average
# -------------------------
y_proba = (w_rf * rf_proba +
           w_xgb * xgb_proba +
           w_lgbm * lgbm_proba)

# Convert to class (threshold = 0.5)
y_pred = (y_proba >= 0.5).astype(int)

# -------------------------
# Evaluation
# -------------------------
print("\n=== Test Set Evaluation (Weighted Ensemble) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Weights: 0.32771535580524347 0.33520599250936334 0.33707865168539325

=== Test Set Evaluation (Weighted Ensemble) ===
Accuracy: 0.9866785079928952
F1 Score: 0.96
ROC AUC: 0.9973065677013045
Confusion Matrix:
 [[931   5]
 [ 10 180]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99       936
           1       0.97      0.95      0.96       190

    accuracy                           0.99      1126
   macro avg       0.98      0.97      0.98      1126
weighted avg       0.99      0.99      0.99      1126



In [22]:
from sklearn.ensemble import VotingClassifier

voting_hard = VotingClassifier(
    estimators=[
        ("rf", best_rf),
        ("xgb", best_xgb),
        ("lgbm", best_lgbm)
    ],
    voting="hard"
)

voting_hard.fit(X_train, y_train)

y_pred = voting_hard.predict(X_test)

print("\n=== Test Set Evaluation (Hard Voting) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 758, number of negative: 3746
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000866 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3809
[LightGBM] [Info] Number of data points in the train set: 4504, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.168295 -> initscore=-1.597760
[LightGBM] [Info] Start training from score -1.597760
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga